# 4— False Positive / False Negative Error Analysis

**False Positive:** Normal → Defective

**False Negative:** Defective → Normal

For manufacturing inspection, false negatives deserve particular attention because a defective product may pass inspection.


In [ ]:
from pathlib import Path
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import datasets,transforms
from torchvision.models import efficientnet_b0

ROOT=Path.cwd(); DATA=ROOT/"data"; MODELS=ROOT/"models"; RESULTS=ROOT/"results"
FP=RESULTS/"false_positives"; FN=RESULTS/"false_negatives"
FP.mkdir(parents=True,exist_ok=True); FN.mkdir(parents=True,exist_ok=True)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")

tf=transforms.Compose([
    transforms.Resize((224,224)),transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225])
])
ds=datasets.ImageFolder(DATA/"test",transform=tf)

model=efficientnet_b0(weights=None)
model.classifier[1]=torch.nn.Linear(model.classifier[1].in_features,2)
model.load_state_dict(torch.load(MODELS/"best_model.pth",map_location=DEVICE))
model.to(DEVICE).eval()


In [ ]:
rows=[]
with torch.no_grad():
    for i,(x,y) in enumerate(ds):
        prob=torch.softmax(model(x.unsqueeze(0).to(DEVICE)),1)
        pred=int(prob.argmax(1).item())
        conf=float(prob.max(1).values.item())
        path,_=ds.samples[i]
        rows.append({"path":path,"true":ds.classes[y],"pred":ds.classes[pred],"confidence":conf})

df=pd.DataFrame(rows)
fp=df[(df.true=="normal")&(df.pred=="defective")].sort_values("confidence",ascending=False)
fn=df[(df.true=="defective")&(df.pred=="normal")].sort_values("confidence",ascending=False)

print("False positives:",len(fp))
print("False negatives:",len(fn))


In [ ]:
def show(frame,title,n=8):
    frame=frame.head(n)
    fig,axes=plt.subplots(2,4,figsize=(14,7)); axes=axes.ravel()
    for ax in axes: ax.axis("off")
    for ax,(_,r) in zip(axes,frame.iterrows()):
        ax.imshow(Image.open(r.path).convert("RGB"))
        ax.set_title(f"True: {r.true}\nPred: {r.pred}\nConf: {r.confidence:.2f}")
        ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

show(fp,"False Positives — Normal predicted as Defective")
show(fn,"False Negatives — Defective predicted as Normal")


In [ ]:
def save(frame,dest,n=8):
    for i,(_,r) in enumerate(frame.head(n).iterrows()):
        src=Path(r.path)
        Image.open(src).convert("RGB").save(dest/f"{i:02d}_{src.name}")

save(fp,FP); save(fn,FN)
print("Saved FP examples:",FP)
print("Saved FN examples:",FN)


## Writing the final error analysis

Inspect the displayed images before making claims.

For **false positives**, check reflections, shadows, texture, orientation and unusual but normal surface appearance.

For **false negatives**, check whether defects are small, subtle, low-contrast, partially hidden, or affected by resizing.

Only report causes that are actually supported by your images.
